# 37 · ML-Decoder trên **đủ 1369 patch token** + fusion EHR theo bệnh

`35` đã đóng phần probe trên embedding đông cứng: loss là **`asl40` @ `E=60`, lấy epoch cuối**;
đầu đọc **phi tuyến điểm-theo-điểm** là null (`MLP − linear = −0.010`); mốc corr cho ensemble là
`0.9060`. Còn mở đúng hai thứ, và notebook này làm cả hai:

* **đọc 1369 token thế nào** — `patch` hiện là **trung bình**, một vector cho mọi nhãn mọi vị trí
* **EHR vào bằng đường nào** — `35` mới chỉ thử `concat`

**Mọi arm chạy trên MỘT forward backbone.** Tham số các arm rời nhau ⇒ `∂(ΣLᵢ)/∂θᵢ = ∂Lᵢ/∂θᵢ`
⇒ tương đương **toán học tuyệt đối** với chạy riêng từng arm (`AMENDMENTS` ①). Điều kiện để
khẳng định đó đúng: **clip gradient phải làm RIÊNG từng arm** — clip chung tính một chuẩn toàn cục
rồi chia đều, tức ghép các arm lại với nhau. `C7` clip riêng và in tỉ lệ kích hoạt.

---

## Ba trục, dùng lẫn nhau

| trục | giá trị | câu hỏi | mục kế hoạch |
|---|---|---|---|
| `tok` | `fam` 9 họ · `k1` 1 vector · **`col` 247 cột + 1 tổng hợp** | chuỗi `K/V` EHR **gồm gì** | **B2b** |
| `mode` | `none` · `cat` = `[patch ; ehr]` · `gate` = attention riêng + `g_c` | **query LỚP** đọc EHR ra sao | **B2c** |
| `t1` | có / không | **PATCH** có được EHR điều biến **theo vùng** không | **B2a** |

`B2a` và `B2c` là hai **bố cục** (ai làm `Q`, ai làm `K/V`). `B2b` **không phải bố cục** — nó là biến
mô tả chuỗi `K/V`, áp cho cả hai.

```
B2a (tang 1)   Q = 1369 PATCH    K/V = EHR            "boi canh khien VUNG NAO dang ngo"
B2c (tang 2)   Q = 40 query LOP  K/V = patch va/hoac EHR   "NHAN NAY co quan tam EHR khong"
```

---

## 12 arm

| arm | `tok` | `mode` | `t1` | vai trò |
|---|---|---|---|---|
| `mean_lin` | — | — | — | **mốc nội bộ**. Mọi so sánh phải nằm trong nb này (ảnh ở đây có tăng cường + mã hoá lại JPEG, nên **không** trùng `0.2093` của `35`) |
| `mean_lin_ehr` | — | — | — | `Q_CTX` concat, đối chiếu `0.2111` của `35` |
| `mldec` | — | `none` | — | **A3** — ML-Decoder không EHR. Cổng `≥ +0.010` |
| `mldec_cat` | `fam` | `cat` | — | bố cục **gốc của kế hoạch**. v1: nó **thắng** `gate` |
| `mldec_gate` | `fam` | `gate` | — | biến thể attention riêng + `g_c` |
| `mldec_gate_zero` | `fam` | `gate` | — | **đối chứng khớp tham số** — EHR thay bằng hằng số học được (`06b` H4z/H6z) |
| `mldec_gate_k1` | `k1` | `gate` | — | control của **B2b**. ⚠ `K=1` ⇒ softmax trên 1 phần tử = 1.0 ⇒ `W_Q`,`W_K` **không có gradient**, `524,288` tham số chết |
| `mldec_col_cat` | `col` | `cat` | — | hướng **A** trong bố cục gốc |
| `mldec_col_gate` | `col` | `gate` | — | hướng **A** trong bố cục gate |
| `mldec_t1` | `col` | `none` | ✓ | **B2a một mình** — tầng 2 không đọc EHR |
| `mldec_t1_zero` | `col` | `none` | ✓ | **đối chứng** cho B2a |
| `mldec_t1t2` | `col` | `gate` | ✓ | cả hai tầng — có cộng dồn không |

---

## Phép so nào trả lời câu nào

```
A3          mldec - mean_lin                        cong +0.010
Case C      lift EHR duoi mean-pool  vs  duoi ML-Decoder     <- o 2x2, mien phi
B2b         col vs fam vs k1      (cung mode=gate)   chuoi K/V nao dung
B2c         cat vs gate           (cung tok)         bo cuc nao dung
B2a         t1 - t1_zero                             tang 1 co gia tri that khong
cong don    t1t2  vs  col_gate  va  t1               hai tang co cong nhau khong
VON         gate_zero - mldec  va  t1_zero - mldec   phan CHI do them tham so
```

Mọi arm có EHR còn phải qua **phép tráo nhân quả** ở `C9`: tráo EHR sang bệnh nhân khác, giữ nguyên
nhãn. mAP tăng mà `corr(logit)` vẫn ~1.0 thì mức tăng **không đến từ EHR** (mốc `06b`: `0.999934`).

---

## Kỷ luật kế thừa từ `35`

`asl40 (γ_neg=4, γ_pos=0)` · **không early-stop, lấy epoch cuối**, cosine trải đúng trên `E` ·
ngưỡng đọc lấy từ **biên độ seed của chính chỉ số đó** (macro `±0.0007`, tail `±0.0011…0.0023`),
**không** dùng `0.005` cho tail.

**Trần đã biết của định tuyến theo bệnh** (`35`, sau khi trừ thiên lệch oracle `+0.0006`):
macro `+0.0019`, và nó **dồn hết vào medium `+0.0028` / tail `+0.0050`; ở head bằng 0**.
⇒ arm B2c phải chấm bằng **tail và medium**, không phải macro.

---

## v1 đã cho gì · v2 đổi gì

```
v1 (E=12, 7 arm, LR chung cho g_c, clip TOAN CUC)
  A3      mldec - mean_lin  +0.0387   gap 4x cong, DEU tren ca ba tier   <- ket qua lon nhat
  Case C  lift 0.0021 -> 0.0036       EHR giup HON duoi ML-Decoder, Case C KHONG dung
  B2c     G_ehr +0.0006 (tran +0.0019)   g_c ket thuc o chi +-0.02..0.04
          corr(logit) trao EHR = 0.9986  -> chu ky "EHR khong duoc dung" cua 06b
  B2b     K=1 (+0.0037) THANG K=9 (+0.0006)
  von     gate_zero - mldec = +0.0031  -> 83% cua "gate - mldec" la do THAM SO
  E=12    moi arm dat dinh ep6-9 roi tut => bang v1 thap hon dinh ~0.004

v2  RUN_TAG='v2_E8_gatelr20' — ghi sang thu muc rieng, v1 con nguyen
  E = 8                    dung dinh do duoc, va re hon
  LR(g_c) x 20             g_c gan nhu khong roi khoi 0 o v1; day la chan doan, khong phai prior
  clip RIENG tung arm      khoi phuc tuong duong AMENDMENTS (1)
  + 5 arm: col_cat, col_gate, t1, t1_zero, t1t2
```

⚠ v1 và v2 khác nhau ở **bốn** thứ cùng lúc (`E`, LR của `g_c`, cách clip, số arm). Nếu dòng
`clip kich hoat` ở `C7` in ra tỉ lệ đáng kể thì **không so trực tiếp v1 với v2 được** — mọi kết luận
phải rút trong nội bộ một lần chạy.

## C0 · Config

In [ ]:
import os, json, math, time, random, shutil, tarfile, io as _io, re
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
# Doi RUN_TAG => ket qua ghi sang thu muc MOI, lan chay cu con nguyen. Khong phai di chuyen
# file, khong the lo tay de len. `resume.pt` cung nam trong do nen C7 bat dau lai tu epoch 0.
RUN_TAG = 'v3_E12_binfix'                 # v1 nam thang trong stage4_nb37/ ; v2 o stage4_nb37/v2_E8_gatelr20/
RES37   = os.path.join(OUT_DIR, 'stage4_nb37', RUN_TAG); os.makedirs(RES37, exist_ok=True)
CKPT    = os.path.join(RES37, 'ckpt');          os.makedirs(CKPT, exist_ok=True)
LOCAL   = '/content/cxrlt_518'                                   # cache anh 518 tren dia LOCAL

W_FRONTAL, W_LATERAL = 0.7, 0.3
SEED    = 42
SIZE    = 518                      # canh dai; RAD-DINO patch 14 -> 37x37 = 1369 token
EPOCHS  = 12                       # Cosine trai DUNG tren E, lay EPOCH CUOI (ky luat 35 §0.1).
                                   # v1 (E=12, clip TOAN CUC, LR g_c thuong): dinh ep6-9 roi TUT.
                                   # v2 (E=8, clip rieng, LR g_c x20): con LEN o ep8 —
                                   #   5/12 arm vuot nguong hoi tu +0.0007, dan dau la `mldec` (+0.0013).
                                   # Hai lan chay KEP diem toi uu. Ly do: cosine u theo E, nen o E=12
                                   # thi ep8 con LR cao (chua u) roi moi overfit; o E=8 thi ep8 DA u xong
                                   # va van con len. Bang chung: `cat` @E=8 epoch cuoi 0.2429 HON
                                   # dinh 0.2414@ep8 cua E=12.  => toi uu quanh E~12 VOI lich u tron.
BATCH, ACCUM = 120, 2              # EFF = BATCH*ACCUM = 240, GIU NGUYEN nhu v1 =>
                                   # lich LR va so buoc optimizer khong doi, chi it lan
                                   # accumulate hon. Cac cap khac giu EFF=240:
                                   #   (48,5) (60,4) (80,3) (120,2) (240,1)
                                   # v1 dung (48,5) va chi cham 13.2/95.6 GB VRAM => GPU RANH.
                                   # VRAM ~ tuyen tinh theo BATCH: (120,2) ~ 33 GB, (240,1) ~ 66 GB.
                                   # 48 la cau hinh 04b do duoc o 518 bf16 khi FINE-TUNE; o day
                                   # backbone dong cung (no_grad) nen nhe hon, nhung co 7 head CO
                                   # gradient. OOM -> ha BATCH, nang ACCUM de giu EFF khong doi.
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
AMP = torch.bfloat16
NW = min(8, (os.cpu_count() or 2))  # 244 anh/s o v1 voi nw=2 => NUT THAT LA GIAI MA JPEG,
                                   # khong phai GPU. Moi worker giai ma 518x518 duoc ~100-200/s.
NOISE_MACRO = 0.0007               # bien do seed do duoc o 35 (epoch-cuoi, 3 seed)
NOISE_TAIL  = 0.0023

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# CHAN SOM: nb nay ~3-4 gio tren GPU va HANG NGAY tren CPU. Roi xuong CPU im lang thi
# chay-all-roi-treo-may se mat ca phien moi biet.
assert dev == 'cuda', ('KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0. '
                       'Chi bo assert nay khi co y chay CPU de kiem cu phap.')
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'thiet bi: {torch.cuda.get_device_name(0)} | VRAM {_vram:.0f} GB')
if _vram < 15:
    print(f'  !! VRAM {_vram:.0f} GB < 15 GB -> ha BATCH xuong 24, nang ACCUM len 10 (giu EFF ~240)')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head', 'medium', 'tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
if os.path.exists(P_VIEW):
    V = (pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
           .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
else:
    V = ORDER[['dicom_id', 'view_group']].copy(); V['use'] = V.view_group != 'unknown'

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL           # lat KHOP / lat CHON, cat theo subject
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]
N_POS_TF = YM[TF_].sum(0)

print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'tier: head {len(TIER["head"])} | medium {len(TIER["medium"])} | tail {len(TIER["tail"])}')
print(f'ngan sach {EPOCHS} epoch, cosine trai dung tren do, LAY EPOCH CUOI (khong early-stop)')
print(f'ghi ket qua vao: {RES37}')
print(f'CPU {os.cpu_count()} nhan -> num_workers {NW} | BATCH {BATCH} x ACCUM {ACCUM} = EFF {BATCH*ACCUM}')
_old_ck = os.path.join(RES37, 'ckpt', 'resume.pt')
print(f'  ckpt hien co: {"CO -> C7 se RESUME" if os.path.exists(_old_ck) else "khong -> C7 bat dau tu epoch 0"}')

## C1 · Nạp EHR — hai tập, và **nhóm cột thành `K` token**

`35` đo: không định tuyến thì `Q_CTX` (157 cột, không một giá trị lâm sàng nào) bằng `Q_FULL` ở macro
và **hơn** ở tail; có định tuyến thì `Q_FULL` có trần cao hơn `+0.0010`. Nên giữ **cả hai**.

`K` token, mỗi token một **họ cột** (`Q_FULL` → 9 họ, `Q_CTX` → 4). Đây là điểm khiến chẩn đoán **đọc được**: sau khi huấn luyện
ta biết *"nhãn Pneumonia dồn bao nhiêu attention lên token vi sinh"*. `K=1` (gộp hết) là **control**:
nếu `K` họ không hơn `K=1` thì việc tách theo họ không mua được gì.

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()

def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = {'Q_CTX': [c for c in SPEC['Q_CTX'] if c in E_DF.columns],
        'Q_FULL': [c for c in SPEC['Q_FULL'] if c in E_DF.columns]}
EHR = {k: _mat(v) for k, v in COLS.items()}
# chuan hoa bang thong ke cua LAT KHOP — khong cham lat chon, khong cham val
EHR_MU, EHR_SD = {}, {}
EHR_CLIP = 10.0     # cot NHI PHAN cuc hiem lam chuan hoa no ra gia tri KHONG LO:
                    #   1 duong / 219,785  ->  sd 2.1e-3  ->  gia tri chuan hoa 468
                    # Probe tuyen tinh (35) chiu duoc — chi can hoc trong so nho. ATTENTION thi
                    # KHONG: `EHRColTok` dua thang x_j vao token (T_j = b_j + x_j*W_j), va score
                    # ti le voi M^2 nen bien do 30 cho score ~2e4 -> softmax bao hoa -> inf/NaN.
BIN_OF = {}
for k, m in EHR.items():
    # Cot NHI PHAN thi KHONG z-score. z-score bien gia tri "co" thanh sqrt((1-p)/p):
    #   p = 1/219,785 -> 468   |   p = 0.01 -> 9.9   |   p = 0.5 -> 1.0
    # Do chinh la nguon cua gia tri khong lo, va no la LOI BIEU DIEN chu khong phai
    # loi cua du lieu: cot chi bao khong co "do lech chuan" de chia.
    # `29b` one-hot top-K vi khuan (`micro_org_*`) + co comorbidity => rat nhieu cot cuc hiem.
    b_ = np.array([np.isin(np.unique(m[:, j]), (0.0, 1.0)).all() for j in range(m.shape[1])])
    BIN_OF[k] = b_
    mu = m[TF_].mean(0, keepdims=True); sd = m[TF_].std(0, keepdims=True) + 1e-6
    mu[:, b_] = 0.0; sd[:, b_] = 1.0                       # nhi phan: giu nguyen 0/1
    EHR_MU[k], EHR_SD[k] = mu, sd
    z = (m - mu) / sd
    _n_out = int((np.abs(z) > EHR_CLIP).sum())
    EHR[k] = np.clip(z, -EHR_CLIP, EHR_CLIP).astype('float32')  # kep chi con la chan cuoi
    print(f'  {k}: {int(b_.sum())}/{len(b_)} cot NHI PHAN (giu 0/1) | '
          f'|z| max {np.abs(z).max():6.1f} | kep {_n_out:,} o ({_n_out/z.size:.4%})')

# --- gom cot thanh 8 HO. Quy tac, khong phai tay; moi cot phai roi vao DUNG mot ho. ---
FAM_RE = [('ctx',    r'^ctx_'),
          ('demo',   r'^(age|gender)$'),   # nhan khau hoc: LUON co mat, khong phai phep do.
                                           # `29b` xep vao "gia tri" vi muc dich cua no la hoi
                                           # "loai benh nhan vs benh"; o day chung la HO rieng.
          ('mask',   r'_mask$'),
          ('n_lab',  r'^n_lab_'),
          ('dh_lab', r'^dh_lab_'),
          ('lab',    r'^lab_'),
          ('vital',  r'^vital_'),
          ('como',   r'^como_'),
          ('micro',  r'^micro_')]
def famsplit(cols):
    out, seen = {}, set()
    for nm, rx in FAM_RE:
        g = [i for i, c in enumerate(cols) if re.search(rx, c) and i not in seen]
        seen |= set(g)
        if g: out[nm] = np.array(g)
    du = [c for i, c in enumerate(cols) if i not in seen]
    assert not du, f'cot khong thuoc ho nao: {du[:10]}'
    return out
FAM = {k: famsplit(v) for k, v in COLS.items()}
for k in COLS:
    print(f'{k}: {len(COLS[k])} cot -> {len(FAM[k])} ho  ' +
          ' | '.join(f'{n}:{len(ix)}' for n, ix in FAM[k].items()))

## C2 · Stage ảnh xuống **đĩa local**

Shard trên Drive là JPEG letterbox **1022**. Giải mã → resize `518` → mã hoá lại một lần, còn ~11 GB
local; mọi epoch sau đọc NVMe thay vì Drive (đọc tar trực tiếp trên mount FUSE **treo cứng** — đã dính
ở `33`, xem §4 kế hoạch). **8 luồng**: đơn luồng là ~50 ảnh/s ⇒ 2.1 giờ cho 377k ảnh.

> Việc mã hoá lại JPEG khiến arm `mean_lin` ở đây **không trùng khít** `0.2093` của `35`.
> Đó là lý do `mean_lin` phải là một arm chứ không phải một con số lấy sẵn: **mọi so sánh nội bộ**.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

# Duong dan LAY TU 28c C0 / 31 ARM_CFG._LB — khong doan.
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
assert os.path.isdir(SHARD_DIR), (
    f'khong thay {SHARD_DIR}. Co trong {DATA_DIR}: '
    f'{[d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))][:20]}')
TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
              if f.endswith('.tar') and f.startswith('lb_'))
assert TARS, f'khong co tar `lb_*` trong {SHARD_DIR}'
os.makedirs(LOCAL, exist_ok=True)
DONE = os.path.join(LOCAL, '_done.json')
done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
todo = [t for t in TARS if os.path.basename(t) not in done]
print(f'{len(TARS)} tar tren Drive | da stage {len(done)} | con {len(todo)}')

N_THREAD = 8          # decode+resize+encode DON LUONG la ~50 anh/s => 2.1 gio cho 377k anh.
                      # PIL nha GIL trong phan lon decode/encode nen thread thuc su co tac dung.

def _one(payload):
    name, blob = payload
    dst = os.path.join(LOCAL, os.path.basename(name))
    if os.path.exists(dst): return 0
    im = Image.open(_io.BytesIO(blob)).convert('L')
    im.resize((SIZE, SIZE), Image.BICUBIC).save(dst, 'JPEG', quality=95)
    return 1

t0 = time.time(); n_new = 0
with ThreadPoolExecutor(N_THREAD) as pool:
    for tp in tqdm(todo, desc='stage', unit='tar'):
        with open(tp, 'rb') as f: buf = f.read()      # doc BULK; lap thang tren mount FUSE se TREO
        with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
            jobs = [(m.name, tar.extractfile(m).read())
                    for m in tar.getmembers() if m.name.endswith('.jpg')]
        n_new += sum(pool.map(_one, jobs))
        del buf, jobs
        done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))   # marker SAU khi ghi
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,} | moi ghi {n_new:,} | {(time.time()-t0)/60:.1f} phut')
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh -> chay lai cell nay'
_s = [os.path.getsize(os.path.join(LOCAL, f)) for f in os.listdir(LOCAL)[:2000] if f.endswith('.jpg')]
print(f'~{np.mean(_s)*_n/1e9:.1f} GB tren dia local')

## C3 · Backbone đông cứng + `Dataset`

Tăng cường **có** — `RandomResizedCrop(0.85–1.0)` + xoay `7°`, giống `06b`. Đó là điều biện minh cho
việc tính lại backbone mỗi epoch: nếu không tăng cường thì 1369 token là **hằng số** và ta đang lặp lại
đúng một phép tính 12 lần.

In [ ]:
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'backbone dong cung | hidden {D_CXR} | patch {backbone.config.patch_size}')

class DS(Dataset):
    '''`ehr_perm`: hoan vi hang EHR cho phep TRAO NHAN QUA. Nhan y KHONG bao gio di theo hoan vi.'''
    def __init__(s, idx, tf, ehr_perm=None):
        s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        j = i if s.perm is None else int(s.perm[k])
        return dict(image=s.tf(Image.merge('RGB', (im, im, im))),
                    y=torch.from_numpy(YM[i]), row=i, erow=j)

def loader(idx, tf, shuffle, bs, perm=None, nw=2):
    # `persistent_workers=True` CHI an toan khi loader duoc tao MOT LAN roi dung lai (xem
    # DL_TRAIN o C7). Tao lai moi epoch VA giu worker la nguon spam
    # 'can only test a child process' da thay o v1.
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def forward_backbone(img):
    '''MOT forward dung chung cho MOI arm (06b C7). Tham so cac arm roi nhau nen day la
       tuong duong TOAN HOC TUYET DOI voi chay rieng tung arm, khong phai xap xi.'''
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        h = backbone(pixel_values=img).last_hidden_state          # [B, 1+1369, 768]
    return h[:, 0].float(), h[:, 1:].float()

_b = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
_c, _p = forward_backbone(_b['image'].to(dev))
N_TOK = _p.shape[1]                       # dung bien, KHONG viet cung 1369 (doi SIZE la sai het)
print(f'cls {tuple(_c.shape)} | patch {tuple(_p.shape)}  <- {N_TOK} token')
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'VRAM cho [128, {N_TOK}, {D_CXR}] bf16 = {128*N_TOK*D_CXR*2/1e6:.0f} MB')

## C4 · Kiến trúc

**ML-Decoder** (Ridnik et al., WACV 2023) — `C` query lớp **cross-attend** chuỗi token. Bỏ
self-attention giữa các query ⇒ chi phí **tuyến tính theo số lớp** (`40 × 1369`, không phải
`1369² = 1.87M`). `query_embed.weight.requires_grad = False` là **cố ý** theo bài gốc.

**Hai cách đưa EHR vào, và chúng khác nhau về bản chất:**

```
mldec_cat    K/V = [1369 patch ; K token EHR]      softmax tren 1369+K
             -> EHR chiem ~K/1377 = 0.6% khoi luong attention luc init
             -> DUNG bo cuc H6 ma 06b da bac (truyen token->logit chi 0.0092)

mldec_gate   nhanh anh : CrossAttn(Q=query, K/V=patch)    softmax tren 1369
             nhanh EHR : CrossAttn(Q=query, K/V=ehr)      softmax tren K  <- KHONG the bi nhan chim
             h = h_img + g_c * Proj(h_ehr)                g_c: MOT scalar MOI LOP, init = 0
```

`g_c` init `0` ⇒ khởi động **đúng bằng** CXR-only, nên `gain < 0` không thể do adapter làm nhiễu.
Và sau huấn luyện `g_c` **chính là** định tuyến theo bệnh, đọc thẳng ra được — không cần suy diễn từ
attention. Đó là câu trả lời trực tiếp cho *"nhãn nào cần EHR"*, và `35` đã đo trần của nó:
macro `+0.0019`, medium `+0.0028`, **tail `+0.0050`**, head **`0`**.

In [ ]:
class GroupFC(nn.Module):
    '''Moi lop mot vector trong so rieng -> 1 logit. Khong chia se giua cac lop.'''
    def __init__(s, C, d):
        super().__init__(); s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b        # [B, C, d] -> [B, C]

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a)
        return s.n2(h + s.ff(h)), w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = chuoi token.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, ehr='none', d_ehr_in=0, K=8,
                 t1=False):
        super().__init__()
        s.mode = ehr; s.K = K
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d)
        s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None      # B2a — dat TRUOC tang 2
        if ehr != 'none' or t1:
            s.ein = nn.Linear(d_ehr_in, d)
            if ehr == 'gate':
                s.ehr = XAttn(d, nhead, ff, drop)
                s.pe  = nn.Linear(d, d)                # KHOI TAO BINH THUONG — xem ghi chu duoi
                s.g = nn.Parameter(torch.zeros(C))     # cong TUNG LOP, init 0 (mau ReZero)
                # KHONG zero-init CA HAI. `h + g*pe(he)` voi g=0 VA pe=0 la diem yen ngua chet:
                #   dL/dg  = pe(he) = 0   va   dL/dpe = g = 0   -> nhanh EHR khong bao gio chay.
                # Dry-run da bat dung loi nay (g_c khong nhuc nhich sau 3 epoch).
                # Chi `g=0` la du de khoi dong DUNG BANG CXR-only, va khi do dL/dg = pe(he) != 0.
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        B = patch.shape[0]
        q  = s.q.weight.unsqueeze(0).expand(B, -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None:                              # TANG 1: patch duoc EHR dieu bien
            kv = s.t1(kv, s.ein(etok))                    # theo VUNG, truoc khi query lop doc
        if s.mode == 'cat':
            kv = torch.cat([kv, s.ein(etok)], 1)                 # softmax tren 1369+K
            h, w = s.img(q, kv, need_w); return s.out(h), w
        h, w = s.img(q, kv, need_w)
        if s.mode == 'gate':
            he, we = s.ehr(q, s.ein(etok), need_w)               # softmax CHI tren K token EHR
            h = h + s.g.view(1, -1, 1) * s.pe(he)
            return s.out(h), (w, we)
        return s.out(h), w

class EHRTok(nn.Module):
    '''K token EHR. `fam`: dict ho -> chi so cot. mode `zero` = vector HANG SO hoc duoc,
       khop tham so voi ban that (doi chung H4z/H6z cua 06b).'''
    def __init__(s, fam, d=256, const=False):
        super().__init__()
        s.fam, s.const = fam, const
        s.enc = nn.ModuleList([nn.Sequential(nn.Linear(len(ix), d), nn.GELU(), nn.Linear(d, d))
                               for ix in fam.values()])
        s.idx = [torch.as_tensor(ix) for ix in fam.values()]
        if const: s.c = nn.Parameter(torch.randn(len(fam), d) * 0.02)
        s.d = d
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        return torch.stack([m(e[:, ix.to(e.device)]) for m, ix in zip(s.enc, s.idx)], 1)

class EHRColTok(nn.Module):
    '''247 token COT (FT-Transformer: T_j = b_j + x_j*W_j) + 1 token TONG HOP tu MLP nhin
       CA 247 cot cung luc.

       Tinh chat quyet dinh: chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don
       het khoi luong len token tong hop la tai lap duoc k1. `EHRTok` theo HO (K=9) KHONG co
       tinh chat do (khong ho nao chua ca 247 cot), va do chinh la ly do no THUA k1 o v1.

       Gop dung hai thu ma K=1 va K=9 moi ben chi co mot:
         token cot     -> CHON LOC tung cot ("Pneumonia doc cot bach cau", khong phai ca ho)
         token tong hop-> TUONG TAC CHEO (attention la to hop LOI, khong tao duoc tich cot)

       HUONG A: moi cot LUON co token. Cot thieu -> (0-mu)/sigma, mot DIEM CO DINH tren duong
       thang cua cot do, mo hinh cham diem thap cho no duoc. Khong `key_padding_mask` => khong
       co kieu loi 'hang toan -inf -> NaN' voi 11,839 study khong co ban ghi nao.'''
    def __init__(s, d_in, d=256, const=False):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d)   # chan bien do token: `col` dua thang x_j vao token nen mot cot
                                 # hiem co the lan at 247 token con lai. `EHRTok` khong can vi
                                 # gia tri cua no da di qua GELU.
        s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)   # doi chung: HANG SO hoc duoc
    def forward(s, e):
        if s.const: return s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)     # [B, 247, d]
        return s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))       # [B, 248, d]

class Tier1(nn.Module):
    '''B2a — Q = 1369 PATCH, K/V = token EHR. FiLM BIEN THIEN THEO KHONG GIAN.

       softmax chi tren token EHR nen cac patch KHONG canh tranh voi nhau: moi patch tu hoi
       EHR rieng, va EHR khong the bi 1369 patch nhan chim (do la co che that bai cua 06b H6).
       Tra loi 'boi canh khien VUNG NAO dang ngo' — truc KHONG GIAN, khac han truc NGU NGHIA
       cua tang 2 ('NHAN NAY co quan tam EHR khong').

       `pr` ZERO-INIT => khoi dong DUNG BANG CXR-only. CHI MOT he so zero-init: khac v1, noi
       `g_c=0` VA `pe=0` tao diem yen ngua chet (dL/dg = pe(he) = 0 va dL/dpe ~ g = 0).
       O day dL/dpr = dL/dpatch' != 0 ngay tu buoc dau.

       Chi phi: 1369 x 248 attention = 18% mot lop self-attention (1369^2 = 1.87M).
       `need_weights=False` cho phep duong SDPA hop nhat -> khong hien thuc hoa ma tran [1369, 248].'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n  = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(a)

class MeanLin(nn.Module):
    def __init__(s, C, d_in, d_ehr=0):
        super().__init__(); s.fc = nn.Linear(d_in + d_ehr, C); s.use_e = d_ehr > 0
    def forward(s, patch, e=None):
        h = patch.mean(1)
        return s.fc(torch.cat([h, e], 1) if s.use_e else h), None

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

## C5 · Bảy arm + cổng `assert` trước khi tốn một giây GPU nào

In [ ]:
D_MLD, D_ETOK = 512, 256
# Ba TRUC doc lap, dung lan nhau:
#   tok  : chuoi K/V EHR gom gi     fam(9 ho) | k1(1 vector) | col(247 cot + 1 tong hop)   <- B2b
#   mode : query LOP doc EHR the nao  none | cat([patch;ehr]) | gate(attention rieng + g_c)  <- B2c
#   t1   : PATCH co duoc EHR dieu bien theo vung khong                                        <- B2a
ARMS = {
  'mean_lin':        dict(head='mean', ehr=None),
  'mean_lin_ehr':    dict(head='mean', ehr='Q_CTX'),
  'mldec':           dict(head='mld',  ehr=None,     mode='none'),
  # --- v1: giu nguyen de doi chieu ---
  'mldec_cat':       dict(head='mld',  ehr='Q_FULL', mode='cat',  tok='fam'),
  'mldec_gate':      dict(head='mld',  ehr='Q_FULL', mode='gate', tok='fam'),
  'mldec_gate_zero': dict(head='mld',  ehr='Q_FULL', mode='gate', tok='fam', const=True),
  'mldec_gate_k1':   dict(head='mld',  ehr='Q_FULL', mode='gate', tok='k1'),
  # --- huong A: 247 cot + 1 tong hop (B2b) ---
  'mldec_col_cat':   dict(head='mld',  ehr='Q_FULL', mode='cat',  tok='col'),
  'mldec_col_gate':  dict(head='mld',  ehr='Q_FULL', mode='gate', tok='col'),
  # --- B2a: tang 1, Q = patch ---
  'mldec_t1':        dict(head='mld',  ehr='Q_FULL', mode='none', tok='col', t1=True),
  'mldec_t1_zero':   dict(head='mld',  ehr='Q_FULL', mode='none', tok='col', t1=True, const=True),
  'mldec_t1t2':      dict(head='mld',  ehr='Q_FULL', mode='gate', tok='col', t1=True),
}
# `k1`: gop CA vector EHR thanh MOT token. Softmax tren 1 phan tu = 1.0 => moi query lop nhan
# CUNG mot vector => suy bien thanh BIAS TUNG LOP. Neu K=8 khong hon K=1 thi viec tach theo HO
# khong mua duoc gi, va bang [40 x 8] o C9-bis chi la hoa van.

def build_arms(seed=SEED):
    torch.manual_seed(seed)
    M, E = nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        if cfg['head'] == 'mean':
            M[tag] = MeanLin(C, D_CXR, EHR[cfg['ehr']].shape[1] if cfg['ehr'] else 0)
        else:
            K = 0
            if cfg['ehr']:
                n_in, tok, cst = EHR[cfg['ehr']].shape[1], cfg.get('tok', 'fam'), cfg.get('const', False)
                if tok == 'col':
                    E[tag] = EHRColTok(n_in, D_ETOK, cst);            K = n_in + 1
                else:
                    fam = {'all': np.arange(n_in)} if tok == 'k1' else FAM[cfg['ehr']]
                    E[tag] = EHRTok(fam, D_ETOK, cst);                K = len(fam)
            M[tag] = MLDecoder(C, D_CXR, D_MLD, ehr=cfg['mode'], d_ehr_in=D_ETOK, K=K,
                               t1=cfg.get('t1', False))
    return M.to(dev), E.to(dev)

MOD, ETK = build_arms()
print(f'{"arm":<17} {"tham so":>12}')
for tag in ARMS:
    n = sum(p.numel() for p in MOD[tag].parameters() if p.requires_grad)
    n += sum(p.numel() for p in ETK[tag].parameters()) if tag in ETK else 0
    print(f'  {tag:<15} {n:>12,}')

# --- cong 1: backbone khong co gradient nao ---
assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
# --- cong 2: query lop KHONG hoc (thiet ke cua bai goc, dung "sua") ---
for t in ('mldec', 'mldec_gate'):
    assert not MOD[t].q.weight.requires_grad, f'{t}: query_embed dang hoc'
# --- cong 3: gate zero-init => mldec_gate PHAI cho logit y het mldec_gate voi g=0 ---
with torch.no_grad():
    _p = torch.randn(4, N_TOK, D_CXR, device=dev)
    _e = torch.randn(4, EHR['Q_FULL'].shape[1], device=dev)
    _t = ETK['mldec_gate'](_e)
    MOD['mldec_gate'].eval()
    l1, _ = MOD['mldec_gate'](_p, _t)
    _g = MOD['mldec_gate'].g.clone(); MOD['mldec_gate'].g.data.fill_(0.0)
    l2, _ = MOD['mldec_gate'](_p, _t)
    MOD['mldec_gate'].g.data.copy_(_g)
    d = (l1 - l2).abs().max().item()
assert d < 1e-5, f'gate KHONG zero-init: lech {d:.2e} -> gain<0 se khong phan biet duoc EHR vo dung voi adapter gay nhieu'
print(f'\ncong: backbone dong cung OK | query co dinh OK | gate zero-init OK (lech {d:.1e})')

# --- cong 5: Tier1 zero-init => arm co t1 PHAI cho logit y het khi bo tang 1 ---
with torch.no_grad():
    _t = ETK['mldec_t1'](_e); MOD['mldec_t1'].eval()
    l3, _ = MOD['mldec_t1'](_p, _t)
    _keep = MOD['mldec_t1'].t1; MOD['mldec_t1'].t1 = None
    l4, _ = MOD['mldec_t1'](_p, _t)
    MOD['mldec_t1'].t1 = _keep
    d1 = (l3 - l4).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'cong: Tier1 zero-init OK (lech {d1:.1e}) — khoi dong dung bang CXR-only')

# --- cong 6: chuoi EHR cua arm `col` phai dai 247+1 ---
with torch.no_grad():
    _nc = ETK['mldec_col_gate'](_e).shape[1]
assert _nc == EHR['Q_FULL'].shape[1] + 1, f'chuoi col = {_nc}, phai la {EHR["Q_FULL"].shape[1]}+1'
print(f'cong: chuoi col = {_nc} token ({_nc-1} cot + 1 tong hop) OK — CHUA k1 nhu truong hop rieng')
# --- cong 7: doi chung khop tham so ---
n1 = sum(p.numel() for p in ETK['mldec_gate'].parameters())
n0 = sum(p.numel() for p in ETK['mldec_gate_zero'].parameters())
print(f'doi chung: gate {n1:,} vs gate_zero {n0:,} tham so EHR '
      f'({"KHOP" if abs(n1-n0)/max(n1,1) < 0.5 else "LECH — G se lan ca hieu ung dung luong"})')

## C6 · Chấm điểm (giống `32` C2 / `35` C3) + AP tie-aware

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V.view_group.to_numpy()[VA] == 'frontal') & V.use.to_numpy()[VA]
_isL = (V.view_group.to_numpy()[VA] == 'lateral') & V.use.to_numpy()[VA]

def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Ys = YM[VA][_first]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

def score(p):
    ap_i = ap_cols(YM[VA], p)
    ps, ok = pool_study(p)
    ap_s = np.array([average_precision_score(_Ys[ok][:, j], ps[ok][:, j]) for j in range(C)])
    return ap_i, ap_s
print(f'val {VA.sum():,} anh -> {len(_uid):,} study')

## C7 · Vòng lặp — **một forward, bảy arm**, checkpoint mỗi epoch

Không early-stop. Cosine trải đúng trên `EPOCHS`, lấy **epoch cuối** (`35` §0.1). Vẫn ghi val mỗi
epoch để biết đường cong **còn đang lên hay không** — nếu còn thì chạy lại ở `E` lớn hơn, chứ không đoán.

In [ ]:
# ============================ CONG KIEM TRANG THAI ============================
# C7 doc RAT nhieu bien tu C0-C6. Chay lai le te vai cell (rat thuong gap voi notebook)
# de lai trang thai CU ma khong bao gi ca — da dinh: doi RUN_TAG o C0 nhung khong chay lai C0,
# nen C7 di tim resume.pt o thu muc cua LAN CHAY TRUOC va no khi nap.
# Cong nay chay trong 1 giay va chan truoc khi ton GPU.
_g = globals()
_need = ['RES37','CKPT','EPOCHS','BATCH','ACCUM','LR','WD','CLIP','WARMUP','AMP','dev',
         'ARMS','MOD','ETK','EHR','FAM','I_TF','I_VA','YM','C',
         'loader','train_tf','eval_tf','forward_backbone','score','ap_cols','tiers','ASL']
_miss = [n for n in _need if n not in _g]
assert not _miss, f'thieu {_miss} -> chay lai C0..C6 THEO THU TU'
assert set(MOD.keys()) == set(ARMS), (
    f'MOD co {sorted(set(MOD.keys()) ^ set(ARMS))} lech voi ARMS -> chay lai C5')
for _t, _c in ARMS.items():
    assert _c['ehr'] is None or _c['ehr'] in EHR, f'{_t} can EHR[{_c["ehr"]}] -> chay lai C1'
print(f'trang thai: {len(ARMS)} arm | RES37 = .../{os.path.basename(RES37)} | E={EPOCHS} | '
      f'BATCH {BATCH}x{ACCUM} | khop {len(I_TF):,} anh | val {len(I_VA):,}')
# ==============================================================================

crit = ASL(gn=4, gp=0).to(dev)

# Tham so cua TUNG arm, de clip RIENG.
#   `clip_grad_norm_` tren TOAN BO tham so tinh mot chuan toan cuc roi chia deu cho tat ca.
#   Voi n arm doc lap, chuan toan cuc ~ sqrt(n) x chuan mot arm => clip kich hoat THUONG HON
#   va keo CA n arm xuong CUNG LUC. Do la COUPLING giua cac arm, pha vo dung khang dinh
#   "tuong duong toan hoc tuyet doi voi chay rieng" (AMENDMENTS (1)) ma toan bo thiet ke
#   forward-dung-chung dua vao.  Loi nay CO TU v1 (7 arm); v2 co 12 arm nen con nang hon.
#   => clip TUNG arm mot, cung nguong CLIP. Bien dem cho biet no co tung kich hoat khong.
PARAM_OF = {t: [p for m in ([MOD[t]] + ([ETK[t]] if t in ETK else []))
                for p in m.parameters() if p.requires_grad] for t in ARMS}
N_CLIP = {t: 0 for t in ARMS}; N_STEP = 0
G_LR_MULT = 20        # LR rieng cho cong g_c — xem ghi chu duoi

def make_opt():
    '''`g_c` init 0 nen gradient vao no NHO, va o v1 (12 epoch, LR chung) no ket thuc o chi
       +-0.02..0.04 — gan nhu chua roi khoi 0. Hau qua do duoc: G_ehr chi +0.0006 trong khi
       arm K=1 duoc +0.0037, va corr(logit) khi trao EHR la 0.9986 (chu ky "EHR khong duoc
       dung" cua 06b). Cho rieng `g_c` LR cao hon 20x. LambdaLR nhan CHUNG mot he so len moi
       group nen ti le 20x giu nguyen suot lich.
       KHONG dat prior theo nhom benh bang tay: do la chon so cung, va no se ro ri kien thuc
       lay tu val vao thiet ke. `g_c` van hoc tu 0.'''
    gates = [MOD[t].g for t in ARMS if getattr(MOD[t], 'mode', None) == 'gate']
    gid   = {id(p) for p in gates}
    ps  = [p for m in (MOD, ETK) for p in m.parameters()
           if p.requires_grad and id(p) not in gid]
    dec = [p for p in ps if p.ndim > 1]; nod = [p for p in ps if p.ndim <= 1]
    assert gates, 'khong tim thay g_c nao -> kiem lai ten thuoc tinh'
    return torch.optim.AdamW([{'params': dec,   'weight_decay': WD},
                              {'params': nod,   'weight_decay': 0.0},
                              {'params': gates, 'lr': LR * G_LR_MULT, 'weight_decay': 0.0}], lr=LR)

def ehr_batch(tag, erow):
    cfg = ARMS[tag]
    if not cfg['ehr']: return None
    e = torch.from_numpy(EHR[cfg['ehr']][erow.numpy()]).to(dev)
    return e if cfg['head'] == 'mean' else ETK[tag](e)

def run_arms(patch, erow, need_w=False):
    out = {}
    for tag, cfg in ARMS.items():
        e = ehr_batch(tag, erow)
        out[tag] = MOD[tag](patch, e, need_w=need_w) if cfg['head'] == 'mld' else MOD[tag](patch, e)
    return out

@torch.no_grad()
def predict(idx, perm=None, bs=64):
    for m in (MOD, ETK): m.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    i = 0
    for b in tqdm(loader(idx, eval_tf, False, bs, perm, nw=0), desc='val', leave=False):
        _, pt = forward_backbone(b['image'].to(dev, non_blocking=True))
        n = pt.shape[0]
        for t, (lg, _) in run_arms(pt, b['erow']).items():
            P[t][i:i+n] = torch.sigmoid(lg.float()).cpu().numpy()
        i += n
    _bad = {t: int(np.isnan(v).sum()) for t, v in P.items() if np.isnan(v).any()}
    assert not _bad, (f'predict cho ra NaN: {_bad} (tren {len(idx):,} x {C} o). '
                      'Arm nao co ten trong day la arm phan ky.')
    return P

P_LOG = os.path.join(RES37, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
# Tao MOT LAN roi dung lai moi epoch: DataLoader tu xao lai moi lan iter, nen dung lai la
# dung. Tranh spawn/teardown worker moi epoch (nguon spam 'can only test a child process').
DL_TRAIN = loader(I_TF, train_tf, True, BATCH, nw=NW)
opt = make_opt()
steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS
warm  = int(WARMUP * steps)
sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                        0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
start = 0
_rp = os.path.join(CKPT, 'resume.pt')
if os.path.exists(_rp):
    ck = torch.load(_rp, map_location=dev, weights_only=False)
    # Checkpoint PHAI khop tap arm hien tai. Doi RUN_TAG / them arm ma van tro vao ckpt cu
    # thi load_state_dict NO thay vi bo qua — da dinh mot lan.
    _cur, _old = set(MOD.state_dict()), set(ck['mod'])
    if _cur != _old:
        _thieu, _thua = sorted(_cur - _old), sorted(_old - _cur)
        print(f'!! ckpt {_rp}')
        print(f'   co {len(_old)} khoa | mo hinh hien tai {len(_cur)} khoa')
        if _thieu: print(f'   ckpt THIEU (arm moi): {sorted({k.split(".")[0] for k in _thieu})}')
        if _thua:  print(f'   ckpt THUA  (arm da bo): {sorted({k.split(".")[0] for k in _thua})}')
        print('   => ckpt thuoc LAN CHAY KHAC. BO QUA, bat dau tu epoch 0.')
        print(f'   Muon tiep tuc lan chay cu thi dat RUN_TAG o C0 ve dung tag cua no.')
    else:
        MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk'])
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); start = ck['ep'] + 1
        print(f'RESUME tu epoch {start+1}')

for ep in range(start, EPOCHS):
    for m in (MOD, ETK): m.train()
    t0, tot, nb = time.time(), 0.0, 0
    opt.zero_grad(set_to_none=True)
    for k, b in enumerate(tqdm(DL_TRAIN, desc=f'ep{ep+1}/{EPOCHS}', leave=False)):
        _, pt = forward_backbone(b['image'].to(dev, non_blocking=True))    # MOT forward dung chung
        y = b['y'].to(dev)
        # BAT NaN TAI NGUON. De no troi den cuoi epoch thi sklearn chi bao 'Input contains NaN'
        # ma khong noi ARM nao, BUOC nao — vo dung.
        _o = run_arms(pt, b['erow'])
        if not torch.isfinite(pt).all():
            raise RuntimeError(f'ep{ep+1} buoc {k}: BACKBONE cho ra NaN/inf -> nghi anh hong')
        for _t, (_lg, _) in _o.items():
            if not torch.isfinite(_lg).all():
                _gc = MOD[_t].g.detach() if getattr(MOD[_t], 'mode', None) == 'gate' else None
                raise RuntimeError(
                    f'ep{ep+1} buoc {k}: arm `{_t}` cho ra {int((~torch.isfinite(_lg)).sum())}'
                    f'/{_lg.numel()} logit NaN/inf.'
                    + (f'  |g_c| max {_gc.abs().max():.3f}' if _gc is not None else '')
                    + f'  EHR |max| {float(torch.from_numpy(EHR[ARMS[_t]["ehr"]][b["erow"].numpy()]).abs().max()):.1f}'
                      if ARMS[_t]['ehr'] else '')
        loss = sum(crit(lg.float(), y) for lg, _ in _o.values())
        if not torch.isfinite(loss):
            raise RuntimeError(f'ep{ep+1} buoc {k}: loss = {loss.item()} (logit deu huu han)')
        (loss / ACCUM).backward(); tot += loss.item(); nb += 1
        if (k + 1) % ACCUM == 0:
            for _t, _ps in PARAM_OF.items():                      # clip RIENG tung arm
                if float(torch.nn.utils.clip_grad_norm_(_ps, CLIP)) > CLIP: N_CLIP[_t] += 1
            N_STEP += 1
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    P = predict(I_VA)
    for t, p in P.items():                               # luu TRUOC khi cham: crash van con bang chung
        np.save(os.path.join(RES37, f'p_{t}_ep{ep+1}.npy'), p)
    _s = {t: score(p)[0] for t, p in P.items()}          # score() goi MOT lan moi arm
    LOG[str(ep)] = {t: dict(mAP_anh=float(np.nanmean(a)), **tiers(a)) for t, a in _s.items()}
    json.dump(LOG, open(P_LOG, 'w'))
    torch.save(dict(mod=MOD.state_dict(), etk=ETK.state_dict(), opt=opt.state_dict(),
                    sch=sch.state_dict(), ep=ep), _rp)
    _gs = {t: float(MOD[t].g.detach().abs().max()) for t in ARMS
           if getattr(MOD[t], 'mode', None) == 'gate'}
    print('   |g_c| max: ' + ' '.join(f'{t.replace("mldec_","")}:{v:.3f}' for t, v in _gs.items()))
    _fire = {t: N_CLIP[t]/max(N_STEP,1) for t in ARMS}
    if max(_fire.values()) > 0.01:
        print(f'   clip kich hoat: ' + ' '.join(f'{t}:{v:.0%}' for t, v in _fire.items() if v > 0.01))
    _ips = nb * BATCH / max(time.time() - t0, 1e-9)
    print(f'   thong luong {_ips:.0f} anh/s  (v1: 244 anh/s voi BATCH 48, nw 2). '
          f'Khong tang khi BATCH tang => con IO-bound, lever la NW/giai ma chu khong phai BATCH.')
    print(f'ep{ep+1:>2} loss {tot/max(nb,1):.4f} | ' +
          ' '.join(f'{t}:{LOG[str(ep)][t]["mAP_anh"]:.4f}' for t in ARMS) +
          f' | {(time.time()-t0)/60:.1f} ph')
print('xong')

## C8 · Bảng chính + cổng, và **ô 2×2 trả lời Case C**

In [ ]:
LOG = json.load(open(P_LOG))
EP  = max(int(k) for k in LOG)
print(f'doc EPOCH CUOI = ep{EP+1} (khong early-stop, ky luat 35 §0.1)')

R = {}
for t in ARMS:
    ai, asx = score(np.load(os.path.join(RES37, f'p_{t}_ep{EP+1}.npy')))
    R[t] = dict(mAP_anh=float(ai.mean()), mAP_study=float(asx.mean()), **tiers(ai),
                **{f'ap_{l}': v for l, v in zip(LAB40, ai)})
T = pd.DataFrame(R).T
show = ['mAP_anh', 'mAP_study', 'head', 'medium', 'tail']
print('\n=== BANG CHINH (val, epoch cuoi) ===')
print(T[show].to_string(float_format=lambda v: f'{v:.4f}'))

print(f'\n--- duong cong val theo epoch: con dang len khong? ---')
CU = pd.DataFrame({t: {int(k)+1: v[t]['mAP_anh'] for k, v in LOG.items()} for t in ARMS})
print(CU.to_string(float_format=lambda v: f'{v:.4f}'))
_d = CU.iloc[-1] - CU.iloc[-3] if len(CU) >= 3 else CU.iloc[-1] - CU.iloc[0]
print('\n  3 epoch cuoi thay doi: ' + ' | '.join(f'{t} {_d[t]:+.4f}' for t in ARMS))
print(f'  arm nao con >+{NOISE_MACRO:.4f} => CHUA hoi tu, chay lai voi EPOCHS lon hon.')

print('\n=== A3 · CONG ML-Decoder (>= +0.010) ===')
g_a3 = T.loc['mldec', 'mAP_anh'] - T.loc['mean_lin', 'mAP_anh']
print(f'  mldec - mean_lin = {g_a3:+.4f}   {"QUA CONG" if g_a3 >= 0.010 else "DUOI CONG"}')
for k in ('head', 'medium', 'tail'):
    print(f'    {k:<7} {T.loc["mldec", k] - T.loc["mean_lin", k]:+.4f}')

print('\n=== CASE C · o 2x2 (lift EHR duoi HAI phep gop) ===')
l_mean = T.loc['mean_lin_ehr', 'mAP_anh'] - T.loc['mean_lin', 'mAP_anh']
l_mld  = T.loc['mldec_gate',   'mAP_anh'] - T.loc['mldec',    'mAP_anh']
print(f'                 khong EHR    co EHR      lift')
print(f'  mean-pool      {T.loc["mean_lin","mAP_anh"]:.4f}     {T.loc["mean_lin_ehr","mAP_anh"]:.4f}   {l_mean:+.4f}   (35 do +0.0018)')
print(f'  ML-Decoder     {T.loc["mldec","mAP_anh"]:.4f}     {T.loc["mldec_gate","mAP_anh"]:.4f}   {l_mld:+.4f}')
if l_mld < l_mean - NOISE_MACRO:
    print('  => CASE C DUNG: phep gop tot hon da tu va cho ma EHR dang va. Ha ky vong cho 39.')
elif abs(l_mld - l_mean) <= NOISE_MACRO:
    print('  => EHR DOC LAP voi phep gop. Ky vong cho 39 giu nguyen.')
else:
    print('  => EHR giup NHIEU HON duoi ML-Decoder: dinh tuyen theo lop dang lam viec.')

print('\n=== B2c · dinh tuyen, va DOI CHUNG khop tham so ===')
cap = T.loc['mldec_gate_zero', 'mAP_anh'] - T.loc['mldec', 'mAP_anh']
G   = T.loc['mldec_gate', 'mAP_anh'] - T.loc['mldec_gate_zero', 'mAP_anh']
print(f'  dung luong  (gate_zero - mldec)      {cap:+.4f}   <- CHI do them tham so, khong lien quan EHR')
print(f'  G_ehr       (gate - gate_zero)       {G:+.4f}   <- phan THAT SU do noi dung EHR')
for k in ('head', 'medium', 'tail'):
    print(f'    G_ehr {k:<7} {T.loc["mldec_gate", k] - T.loc["mldec_gate_zero", k]:+.4f}'
          + ('   (35: tran o day = +0.0050)' if k == 'tail' else
             '   (35: tran = +0.0028)' if k == 'medium' else '   (35: tran = 0)'))
print(f'\n  bo cuc H6 cua 06b (token EHR CUNG chuoi K/V):')
print(f'    mldec_cat - mldec = {T.loc["mldec_cat","mAP_anh"] - T.loc["mldec","mAP_anh"]:+.4f}'
      f'   so voi gate {G:+.4f}')
_c = T.loc['mldec_cat', 'mAP_anh'] - T.loc['mldec', 'mAP_anh']
if _c > G + NOISE_MACRO:
    print(f'    cat ({_c:+.4f}) HON gate ({G:+.4f}) => du doan "EHR bi nhan chim trong 1369 patch"')
    print('    KHONG dung tren du lieu nay: cong g_c zero-init hoc CHAM hon la dua thang vao K/V.')
elif G > _c + NOISE_MACRO:
    print(f'    gate ({G:+.4f}) HON cat ({_c:+.4f}) => phan tach softmax LA co che.')
else:
    print(f'    cat {_c:+.4f} ~ gate {G:+.4f}: hai bo cuc khong phan dinh duoc.')
k1 = T.loc['mldec_gate_k1', 'mAP_anh'] - T.loc['mldec_gate_zero', 'mAP_anh']
print()
print('  control K=1 (mot token, suy bien thanh bias tung lop):')
_KF = len(FAM['Q_FULL'])
print(f'    G_ehr(K={_KF}) {G:+.4f}  vs  G_ehr(K=1) {k1:+.4f}   chenh {G-k1:+.4f}')
for _k in ('medium', 'tail'):
    _a = T.loc['mldec_gate', _k] - T.loc['mldec_gate_zero', _k]
    _b = T.loc['mldec_gate_k1', _k] - T.loc['mldec_gate_zero', _k]
    print(f'      {_k:<7} K={_KF} {_a:+.4f} | K=1 {_b:+.4f} | chenh {_a-_b:+.4f}')
print(f'    chenh <= {NOISE_MACRO:.4f} => tach theo HO khong mua duoc gi; bang [40 x K] chi la hoa van.')
T.to_csv(os.path.join(RES37, 'main_table.csv'))

## C9 · Cơ chế — **`g_c` đọc thẳng** + phép tráo nhân quả

`06b` đo `corr(logit) = 0.999934` khi tráo EHR sang bệnh nhân khác; đó là bằng chứng mạnh nhất của họ.
Arm này **phải** báo cùng con số. mAP tăng mà `corr` vẫn ~1.0 thì mức tăng **không đến từ EHR**.

`g_c` là cổng từng lớp, init `0`. Sau huấn luyện nó **chính là** định tuyến theo bệnh — so hạng của nó
với lift EHR đã đo ở `35` là phép kiểm chứng, **không** phải để hard-code (làm thế là rò rỉ val vào
thiết kế).

In [ ]:
ck = torch.load(os.path.join(CKPT, 'resume.pt'), map_location=dev, weights_only=False)
MOD.load_state_dict(ck['mod']); ETK.load_state_dict(ck['etk'])
g = MOD['mldec_gate'].g.detach().cpu().numpy()
# `lift_35` phai DOC TU FILE cua 35, khong go tay. Ban truoc go 16/40 nhan (8 duong nhat +
# 8 am nhat) nen 24 nhan con lai ra NaN, va spearman chi chay tren n=16 — dung HAI DAU MUT
# cua phan bo, mot mau BI CHON. `b1_head_v2.csv` co AP tung nhan cho ca 40 nhan.
# (No chay asl40 @E=60 voi chon epoch theo macro, khong phai giao thuc epoch-cuoi; du de
#  lam moc XEP HANG giua cac nhan, khong dung lam con so tuyet doi.)
_P35 = os.path.join(OUT_DIR, 'stage4_nb35', 'b1_head_v2.csv')
if os.path.exists(_P35):
    _B35 = pd.read_csv(_P35).drop_duplicates('ten', keep='last').set_index('ten')
    _a35 = lambda t: _B35.loc[t, [f'ap_{l}' for l in LAB40]].to_numpy('float64')
    LIFT35 = dict(zip(LAB40, _a35('zero_linear') - _a35('cxr_linear')))   # Q_FULL, mean-pool+linear
    print(f'lift_35: doc {len(LIFT35)}/40 nhan tu {os.path.basename(_P35)}')
else:
    LIFT35 = {}
    print(f'!! khong thay {_P35} -> lift_35 rong, moi doi chieu voi 35 se bi bo qua')
GT = pd.DataFrame(dict(nhan=LAB40, g=g,
                       tier=[t for l in LAB40 for t in TIER if LAB40.index(l) in TIER[t]]))
GT['lift_35'] = GT.nhan.map(LIFT35)
print('=== g_c: 10 nhan mo cong EHR nhat / 10 nhan dong nhat ===')
print(pd.concat([GT.nlargest(10, 'g'), GT.nsmallest(10, 'g')])
        .to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
_m = GT.lift_35.notna()
if _m.sum() >= 8:
    from scipy.stats import spearmanr
    r = spearmanr(GT.g[_m], GT.lift_35[_m])
    print(f'\n  spearman(g_c, lift do o 35) = {r.statistic:+.3f}  p={r.pvalue:.3f}  (n={_m.sum()})')
    print('  DUONG => mo hinh tu tim ra dung nhom nhan ma phep do doc lap da chi ra: co che chay dung.')
    print('  ~0    => mo hinh KHONG tim duoc duong dinh tuyen. Do la CHAN DOAN, khong phai mot con so mAP.')
print(f'\n  |g| theo tier: ' + ' | '.join(
    f'{t} {np.abs(g[TIER[t]]).mean():.4f}' for t in ('head','medium','tail')))

# --- trao nhan qua: EHR sang benh nhan khac, NHAN giu nguyen ---
rs = np.random.default_rng(0)
perm = rs.permutation(len(I_VA))
P0, P1 = predict(I_VA), predict(I_VA, perm=I_VA[perm])
print('\n=== TRAO NHAN QUA (EHR sang benh nhan khac, nhan giu nguyen) ===')
for t in ARMS:
    if not ARMS[t]['ehr']: continue
    lg0 = np.log(P0[t].clip(1e-6, 1-1e-6) / (1 - P0[t].clip(1e-6, 1-1e-6)))
    lg1 = np.log(P1[t].clip(1e-6, 1-1e-6) / (1 - P1[t].clip(1e-6, 1-1e-6)))
    r = np.corrcoef(lg0.ravel(), lg1.ravel())[0, 1]
    d = float(np.nanmean(ap_cols(YM[VA], P1[t])) - np.nanmean(ap_cols(YM[VA], P0[t])))
    print(f'  {t:<16} corr(logit) {r:.6f} | mAP khi trao {d:+.4f}   '
          f'{"EHR KHONG duoc dung" if r > 0.9999 else "EHR co tac dong that"}')
print('  moc 06b: corr = 0.999934 (H6) — bang chung manh nhat cua ho rang token EHR khong duoc dung.')

## C9-bis · Ma trận **định tuyến** `[40 nhãn × K họ EHR]`

`g_c` ở C9 nói **mức độ** — nhãn này dùng EHR bao nhiêu. Bảng dưới nói **phần nào** — nhãn này đọc họ
cột nào của EHR. Cả hai đều là **thiết kế đọc thẳng ra được**, không phải suy diễn từ mAP.

`s.ehr` là cross-attention riêng với `Q = 40 query lớp`, `K/V = K token họ`, nên **mỗi nhãn có phân bố
attention riêng** trên `ctx · demo · mask · n_lab · dh_lab · lab · vital · como · micro`.

Đối chiếu với `35` là **kiểm chứng cơ chế**, không phải để hard-code — nhét danh sách nhãn vào kiến
trúc là rò rỉ val vào thiết kế.

In [ ]:
for m in (MOD, ETK): m.eval()
FAMN = list(FAM['Q_FULL'].keys())
acc, nb_ = np.zeros((C, len(FAMN))), 0
with torch.no_grad():
    for k, b in enumerate(loader(I_VA, eval_tf, False, 64, nw=0)):
        if k >= 30: break
        _, pt = forward_backbone(b['image'].to(dev))
        e = ETK['mldec_gate'](torch.from_numpy(EHR['Q_FULL'][b['erow'].numpy()]).to(dev))
        _, (_, we) = MOD['mldec_gate'](pt, e, need_w=True)          # we: [B, C, K]
        acc += we.float().mean(0).cpu().numpy(); nb_ += 1
W = pd.DataFrame(acc / nb_, index=LAB40, columns=FAMN)
W['g_c']  = g
W['tier'] = [t for l in LAB40 for t in TIER if LAB40.index(l) in TIER[t]]
W['ho_manh'] = W[FAMN].idxmax(1)
print('=== 12 nhan mo cong EHR nhat: doc HO NAO ===')
print(W.nlargest(12, 'g_c')[['tier', 'g_c', 'ho_manh'] + FAMN]
        .to_string(float_format=lambda v: f'{v:.3f}'))
print(f'\nmuc DEU = {1/len(FAMN):.3f}. Lech khoi muc do moi la dinh tuyen THAT.')
print(f'do lech khoi deu (TB tren 40 nhan): {np.abs(W[FAMN].to_numpy()-1/len(FAMN)).mean():.4f}')
print('\n=== moi ho duoc nhung nhan nao doc manh nhat ===')
for f in FAMN:
    t4 = W.nlargest(4, f)
    print(f'  {f:<8} ' + ' · '.join(f'{i}({W.loc[i, f]:.2f})' for i in t4.index))
print('\nDOI CHIEU (do doc lap tren val o 35 — KHONG duoc dung trong thiet ke):')
print('  nhiem trung/dich  Pneumonia · Tuberculosis · Edema · Infiltration -> ky vong `micro`/`lab`')
print('  chan thuong       Rib Fracture · Fracture · Pneumothorax          -> ky vong `ctx`/`n_lab`')
print('  man/cau truc      Kyphosis · Emphysema · Fibrosis                 -> ky vong g_c ~ 0')
print('Khop => co che chay dung thiet ke. Khong khop => mo hinh khong tim duoc duong dinh tuyen,')
print('va DO LA CHAN DOAN, khong phai mot con so mAP.')
W.to_csv(os.path.join(RES37, 'disease_routing.csv'))

## C9-ter · **Thước đo có hợp lệ không** — trước khi kết luận `g_c` sai

`lift_35` đo bằng `AP(mean-pool + EHR, linear) − AP(mean-pool, linear)`, tức mức hữu ích của EHR
**dưới phép gộp trung bình**. Nhưng chính notebook này vừa cho thấy ML-Decoder đọc ảnh theo cách khác
hẳn — mỗi nhãn tự chọn vùng. Một nhãn mà lift ở `35` đến từ *"EHR bù lại thứ pooling làm mất"* sẽ có
lift **bằng 0** dưới ML-Decoder, vì ML-Decoder đã tự lấy lại rồi.

Nên `spearman(g_c, lift_35) ≈ 0` có **hai** cách đọc, và ta phải phân định trước khi chọn một:

```
mo hinh khong dinh tuyen duoc          <- ket luan ve MO HINH
huu ich cua EHR DOI theo che do doc     <- ket luan ve THUOC DO
```

`mldec_cat` **không có `g_c`**, nên `lift_cat = AP(cat) − AP(mldec)` là thước đo **độc lập** về
*"EHR giúp nhãn j bao nhiêu dưới ML-Decoder"*. Đó mới là mốc đúng để chấm `g_c`.

> Riêng kết luận *"`g_c` gần như không mở"* thì **không** phụ thuộc `lift_35`: `g_c` kết thúc ở
> `±0.02–0.04`, `G_ehr +0.0006` so với `K=1` được `+0.0037`, và `corr(logit) = 0.9986`. Ba số đó
> độc lập, nên bản sửa LR ở `C7` đúng bất kể cell này ra sao.

In [ ]:
from scipy.stats import spearmanr
DIAG_DIR = RES37          # tro sang thu muc v1 de phan tich lai lan chay CU, khong phai chay lai
M  = pd.read_csv(os.path.join(DIAG_DIR, 'main_table.csv'), index_col=0)
RT = pd.read_csv(os.path.join(DIAG_DIR, 'disease_routing.csv'), index_col=0)
apc = [f'ap_{l}' for l in LAB40]
ap  = lambda t: M.loc[t, apc].to_numpy('float64')

D = pd.DataFrame(dict(
    nhan      = LAB40,
    tier      = [t for l in LAB40 for t in TIER if LAB40.index(l) in TIER[t]],
    g_c       = RT.loc[LAB40, 'g_c'].to_numpy(),
    lift_cat  = ap('mldec_cat')     - ap('mldec'),              # DOC LAP voi g_c
    lift_gate = ap('mldec_gate')    - ap('mldec_gate_zero'),
    lift_k1   = ap('mldec_gate_k1') - ap('mldec_gate_zero'),
    lift_35   = [LIFT35.get(l, np.nan) for l in LAB40]))

def sp(a, b, nm):
    m = D[a].notna() & D[b].notna()
    r = spearmanr(D[a][m], D[b][m])
    print(f'  {nm:<50} rho {r.statistic:+.3f}  p={r.pvalue:.3f}  n={int(m.sum())}')

print('=== (1) THUOC DO CO HOP LE KHONG — cau hoi chinh ===')
sp('lift_cat', 'lift_35', 'lift(ML-Decoder) vs lift(linear @35)')
print('    rho ~ 0    => huu ich cua EHR DOI THEO che do doc anh. `lift_35` KHONG phai moc hop le')
print('                  cho 37, va ket luan "mo hinh khong dinh tuyen duoc" phai RUT LAI.')
print('    rho cao    => thuoc do on dinh giua hai che do; khi do g_c that su tim sai nhom nhan.')

print()
print('=== (2) g_c co dung khong — cham bang moc TRONG CHE DO ===')
sp('g_c', 'lift_cat',  'g_c vs lift(cat)   <- moc DOC LAP, cung che do')
sp('g_c', 'lift_gate', 'g_c vs lift(gate)  (co vong: cung mot lan chay)')
sp('g_c', 'lift_35',   'g_c vs lift_35     (moc CU, khac che do)')

print()
# --- (3) DONG THUAN nhieu arm, KHONG xep theo mot cot ---
# Xep theo MOT cot la sai: `lift_cat` va `lift_k1` doi DAU o nhieu nhan
# (Clavicle Fracture -0.0451 vs +0.0379, dao 0.083). Doi cot thi danh sach khac han.
# Xep theo DONG THUAN giua cac arm doc lap, hien BIEN DO bat dong, va hien n_pos de
# phan biet tin hieu voi nhieu AP tren nhan hiem.
ARM_EHR = [a for a in ('mldec_cat', 'mldec_col_cat', 'mldec_gate', 'mldec_gate_k1',
                       'mldec_t1', 'mldec_t1t2') if a in M.index]
Lm = np.stack([ap(a) - ap('mldec') for a in ARM_EHR])        # [n_arm, 40], CUNG moc `mldec`
D2 = pd.DataFrame(dict(
    nhan   = LAB40,
    tier   = [t for l in LAB40 for t in TIER if LAB40.index(l) in TIER[t]],
    n_pos  = YM[VA].sum(0).astype(int),
    dong_thuan = Lm.mean(0),
    bien_do    = Lm.max(0) - Lm.min(0),
    n_duong    = (Lm > 0).sum(0),                            # bao nhieu / n_arm noi CO LOI
    g_c        = RT.loc[LAB40, 'g_c'].to_numpy(),
    lift_35    = [LIFT35.get(l, np.nan) for l in LAB40]))
D2['tin_cay'] = np.where(D2.bien_do > 2 * D2.dong_thuan.abs(), 'BAT DONG', 'dong thuan')
_c = ['nhan', 'tier', 'n_pos', 'dong_thuan', 'bien_do', 'n_duong', 'lift_35', 'g_c']

print(f'=== (3) {len(ARM_EHR)} arm DONG THUAN: EHR giup / hai nhan nao DUOI ML-Decoder ===')
_ok = D2[D2.tin_cay.eq('dong thuan')]
print(f'\n10 nhan giup nhat (trong so {len(_ok)} nhan cac arm dong thuan):')
print(_ok.nlargest(10, 'dong_thuan')[_c].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
print('\n10 nhan hai nhat:')
print(_ok.nsmallest(10, 'dong_thuan')[_c].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))

_bd = D2[D2.tin_cay.eq('BAT DONG')]
print(f'\n=== {len(_bd)}/40 nhan cac arm BAT DONG (bien do > 2x |trung binh|) — KHONG ket luan duoc ===')
print(_bd.nsmallest(8, 'n_pos')[_c].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
if len(_bd) and len(_ok):
    print(f'\n  n_pos trung vi: bat dong {_bd.n_pos.median():.0f} vs dong thuan {_ok.n_pos.median():.0f}')
    print('  Bat dong tap trung o nhan HIEM => phan lon la nhieu AP tung nhan, khong phai tin hieu.')

print('\n  dong_thuan theo tier: ' + ' | '.join(
    f'{t} {D2.dong_thuan[D2.tier == t].mean():+.4f}' for t in ('head', 'medium', 'tail')))
_r = spearmanr(D2.g_c, D2.dong_thuan)
print(f'\n  rho(g_c, dong_thuan) = {_r.statistic:+.3f}  p={_r.pvalue:.3f}')
print('  <- phep thu SACH nhat con lai cua disease-aware: cham g_c bang DONG THUAN nhieu arm')
print('     thay vi mot arm. Duong va p nho => cong mo dung nhom nhan. ~0 => van khong.')
D.to_csv(os.path.join(DIAG_DIR, 'lift_per_label.csv'), index=False)
D2.to_csv(os.path.join(DIAG_DIR, 'lift_consensus.csv'), index=False)

## C10 · Ghi `REPORT_STAGE4_NB37.md`

In [ ]:
L = []; w = L.append
w('# REPORT · `nb37` — ML-Decoder(1369) + định tuyến EHR theo bệnh'); w('')
w(f'`asl40` · `E={EPOCHS}` · **epoch cuối** · val {VA.sum():,} ảnh / {len(_uid):,} study · '
  f'bảy head trên **một forward backbone**.'); w('')
w('| arm | mAP ảnh | study | head | medium | tail |'); w('|---|---|---|---|---|---|')
for t in ARMS:
    r = T.loc[t]
    w(f'| `{t}` | {r.mAP_anh:.4f} | {r.mAP_study:.4f} | {r["head"]:.4f} | {r["medium"]:.4f} | {r["tail"]:.4f} |')
w('')
w(f'**A3.** `mldec − mean_lin = {g_a3:+.4f}` (cổng `+0.010`).')
w(f'**Case C.** lift EHR: mean-pool `{l_mean:+.4f}` → ML-Decoder `{l_mld:+.4f}`.')
w(f'**B2c.** dung lượng `{cap:+.4f}` · **G_ehr `{G:+.4f}`** '
  f'(tail `{T.loc["mldec_gate","tail"]-T.loc["mldec_gate_zero","tail"]:+.4f}`, trần `35` = `+0.0050`).')
w('')
w('## Những gì kết quả này KHÔNG loại trừ'); w('')
w('- **Số token EHR.** Chỉ `K = 8` (theo họ cột) được thử. `K=1` (control suy biến thành bias toàn '
  'cục) và `K=250` (FT-Transformer, một token mỗi cột) chưa chạy.')
w('- **Tầng 1 (B2a).** `Q=patch, K/V=EHR` — FiLM biến thiên theo không gian — chưa dựng ở đây.')
w('- **Backbone.** RAD-DINO đông cứng, chưa fine-tune (`38`).')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó** ⇒ ảnh gần như đủ theo định '
  'nghĩa. Không con số nào ở đây nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L)
open(os.path.join(RES37, 'REPORT_STAGE4_NB37.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)